# Exploración del CSV inscripciones
Nos toca extraer y analizar cada columna del CSV para después definir cómo validar y corregir los datos. Acá solo se **mira**: el archivo original no se modifica y las correcciones se hacen en DuckDB con los archivos de la carpeta `sql`.

## 1. Leer el CSV

In [1]:
import pandas as pd

In [2]:
df_original = pd.read_csv("./datos/inscripciones.csv")
df = df_original.copy()

Cantidad de filas y columnas (filas, columnas):

In [3]:
df.shape

(2575, 9)

In [4]:
df.head()

,id_inscripcion,id_alumno,nombre_alumno,carrera,materia,fecha_inscripcion,nota_parcial,nota_final,condicion
0,I00001,274,Agustina Ruiz,Analista de Datos,Estadistica,2024-12-25,6.8,8.6,aprobado
1,I00002,88,Lucia Vega,Tecnicatura en Redes,Ingles Tecnico,2025-10-07,4.2,4.0,desaprobado
2,I00003,400,Juan Ramos,Tecnicatura en Redes,Matematica,2025-07-10,42.0,2.6,desaprobado
3,I00004,253,Tomas Medina,Analista de Datos,Base de Datos,2025-08-01,92.0,4.3,desaprobado
4,I00005,380,Mateo Sosa,Analista de Datos,Estadistica,2025-01-05,4.3,8.6,aprobado


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2575 entries, 0 to 2574
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id_inscripcion     2575 non-null   str    
 1   id_alumno          2575 non-null   int64  
 2   nombre_alumno      2575 non-null   str    
 3   carrera            2575 non-null   str    
 4   materia            2575 non-null   str    
 5   fecha_inscripcion  2575 non-null   str    
 6   nota_parcial       2575 non-null   float64
 7   nota_final         2575 non-null   float64
 8   condicion          2420 non-null   str    
dtypes: float64(2), int64(1), str(6)
memory usage: 181.2 KB


## 2. Valores nulos

In [6]:
df.isna().sum()

id_inscripcion         0
id_alumno              0
nombre_alumno          0
carrera                0
materia                0
fecha_inscripcion      0
nota_parcial           0
nota_final             0
condicion            155
dtype: int64

Solo `condicion` tiene vacíos: regla `condicion_nula`.

## 3. Ids repetidos

In [7]:
df["id_inscripcion"].duplicated().sum()

np.int64(75)

In [8]:
df[df["id_inscripcion"].duplicated(keep=False)].sort_values("id_inscripcion").head(6)

,id_inscripcion,id_alumno,nombre_alumno,carrera,materia,fecha_inscripcion,nota_parcial,nota_final,condicion
20,I00021,386,FRANCO RAMOS,Analista de Datos,Matematica,2024-12-29,8.5,4.1,desaprobado
602,I00021,386,FRANCO RAMOS,Analista de Datos,Matematica,2024-12-29,8.5,4.1,desaprobado
70,I00071,170,Martin Gomez,Tecnicatura en Software,Ingles Tecnico,2024-04-25,9.2,68.0,aprobado
558,I00071,170,Martin Gomez,Tecnicatura en Software,Ingles Tecnico,2024-04-25,9.2,68.0,aprobado
582,I00128,220,Franco Medina,Tecnicatura en Redes,Sistemas Operativos,2024-06-20,65.0,3.0,desaprobado
130,I00128,220,Franco Medina,Tecnicatura en Redes,Sistemas Operativos,2024-06-20,65.0,3.0,desaprobado


Hay inscripciones repetidas: regla `id_inscripcion_duplicado`.

## 4. Nombres con espacios sobrantes

In [9]:
(df["nombre_alumno"] != df["nombre_alumno"].str.strip()).sum()

np.int64(138)

Regla `nombre_alumno_con_espacios`.

## 5. Fechas

In [10]:
df["fecha_inscripcion"].str.contains("/").sum()

np.int64(257)

In [11]:
df[df["fecha_inscripcion"].str.contains("/")]["fecha_inscripcion"].head()

12    09/06/2024
15    02/01/2025
24    05/07/2025
32    30/10/2024
38    17/09/2024
Name: fecha_inscripcion, dtype: str

Algunas fechas vienen como DD/MM/YYYY en lugar de YYYY-MM-DD: regla `fecha_formato_invalido`.

## 6. Notas

In [12]:
df[["nota_parcial", "nota_final"]].describe()

,nota_parcial,nota_final
count,2575.000000,2575.000000
mean,8.643573,8.950136
std,12.604921,13.129281
min,2.000000,2.000000
25%,4.100000,4.100000
50%,6.200000,6.200000
75%,8.300000,8.400000
max,99.000000,99.000000


In [13]:
(df["nota_parcial"] > 10).sum(), (df["nota_final"] > 10).sum()

(np.int64(131), np.int64(154))

Hay notas mayores a 10 (por ejemplo 42): perdieron el punto decimal. Reglas `nota_parcial_fuera_de_rango` y `nota_final_fuera_de_rango`.

## 7. Columna condicion

In [14]:
df["condicion"].value_counts(dropna=False)

condicion
desaprobado     1154
aprobado        1138
NaN              155
APROBADO          27
 APROBADO         26
 DESAPROBADO      21
DESAPROBADO       20
APROBADO          18
DESAPROBADO       16
Name: count, dtype: int64

## 8. Carreras y materias

In [15]:
df["carrera"].value_counts()

carrera
Analista de Datos          873
Tecnicatura en Software    872
Tecnicatura en Redes       830
Name: count, dtype: int64

In [16]:
df["materia"].value_counts()

materia
Matematica             617
Ingles Tecnico         436
Base de Datos          425
Estadistica            235
Sistemas Operativos    225
Python para Datos      215
Redes 1                214
Programacion 1         208
Name: count, dtype: int64

## 9. Resultado del pipeline en DuckDB
Primero hay que ejecutar `python pipeline.py` (o el DAG en Airflow) para que exista `tp_instituto.duckdb`. El código de la extracción está en `sql/extract.sql`.

In [17]:
import duckdb
con = duckdb.connect("tp_instituto.duckdb", read_only=True)

In [18]:
con.execute("SELECT * FROM raw_inscripciones LIMIT 5").df()

,id_inscripcion,id_alumno,nombre_alumno,carrera,materia,fecha_inscripcion,nota_parcial,nota_final,condicion
0,I00001,274,Agustina Ruiz,Analista de Datos,Estadistica,2024-12-25,6.8,8.6,aprobado
1,I00002,88,Lucia Vega,Tecnicatura en Redes,Ingles Tecnico,2025-10-07,4.2,4.0,desaprobado
2,I00003,400,Juan Ramos,Tecnicatura en Redes,Matematica,2025-07-10,42.0,2.6,desaprobado
3,I00004,253,Tomas Medina,Analista de Datos,Base de Datos,2025-08-01,92.0,4.3,desaprobado
4,I00005,380,Mateo Sosa,Analista de Datos,Estadistica,2025-01-05,4.3,8.6,aprobado


In [19]:
con.execute("""
SELECT 'raw_inscripciones' AS tabla, COUNT(*) AS filas FROM raw_inscripciones
UNION ALL SELECT 'validos', COUNT(*) FROM validos
UNION ALL SELECT 'rechazados', COUNT(*) FROM rechazados
UNION ALL SELECT 'rechazados_limpios', COUNT(*) FROM rechazados_limpios
UNION ALL SELECT 'tabla_final', COUNT(*) FROM tabla_final
""").df()

,tabla,filas
0,raw_inscripciones,2575
1,validos,1525
2,rechazados,1050
3,rechazados_limpios,1050
4,tabla_final,2500


In [20]:
con.execute("SELECT motivo_rechazo, COUNT(*) AS cantidad FROM rechazados GROUP BY motivo_rechazo ORDER BY cantidad DESC LIMIT 10").df()

,motivo_rechazo,cantidad
0,fecha_formato_invalido,243
1,nota_final_fuera_de_rango,146
2,condicion_nula,145
3,nombre_alumno_con_espacios,126
4,condicion_no_permitida,122
5,nota_parcial_fuera_de_rango,119
6,id_inscripcion_duplicado,87
7,id_inscripcion_duplicado | fecha_formato_invalido,14
8,id_inscripcion_duplicado | nota_parcial_fuera_...,12
9,id_inscripcion_duplicado | nombre_alumno_con_e...,12


## 10. Respuestas a las consultas

In [21]:
con.execute("SELECT * FROM consulta_1_nota_promedio_por_materia").df()

,materia,nota_final_promedio
0,Estadistica,6.06
1,Matematica,6.05
2,Sistemas Operativos,6.05
3,Base de Datos,5.99
4,Ingles Tecnico,5.98
5,Python para Datos,5.96
6,Programacion 1,5.83
7,Redes 1,5.76


In [22]:
con.execute("SELECT * FROM consulta_2_aprobacion_por_carrera").df()

,carrera,porcentaje_aprobacion
0,Analista de Datos,53.83
1,Tecnicatura en Redes,49.56
2,Tecnicatura en Software,44.22


In [23]:
con.execute("SELECT * FROM consulta_3_top5_alumnos").df()

,id_alumno,inscripciones,promedio_nota_final
0,244,5,8.52
1,157,7,7.90
2,15,7,7.89
3,58,6,7.88
4,245,6,7.85


In [24]:
con.execute("SELECT * FROM consulta_4_inscripciones_por_mes").df()

,mes,inscripciones
0,2024-01,114
1,2024-02,102
2,2024-03,98
3,2024-04,109
4,2024-05,103
5,2024-06,119


In [25]:
con.execute("SELECT * FROM consulta_5_materia_con_mas_desaprobados").df()

,materia,desaprobados
0,Matematica,293


In [26]:
con.close()